# MicroStrategy REST Admin

## Load libraries

In [30]:
import yaml
import json
import pandas as pd
from mstrio.object_management import folder
from mstrio.api import metrics,browsing,filters,attributes,transformations

from mstr_robotics.mstr_classes import MdSearches
from mstr_robotics._connectors import MstrApi

from time import sleep


all_comp_obj_d_l=[]
i_md_searches=MdSearches()
i_mstr_api=MstrApi()

In [31]:
from mstr_robotics.mstr_classes import get_conn
with open('..\\config\\user_d.yml', 'r') as openfile:
    user_d = yaml.safe_load(openfile)

conn_params =  user_d["conn_params"]
conn = get_conn(**conn_params)
conn.headers['Content-type'] = "application/json"
project_id=user_d["conn_params"]["project_id"]
conn.select_project(project_id)

Connection to Strategy One Intelligence Server has been established.
Project selected in Connection object:
Project object named: 'MicroStrategy Tutorial' with ID: 'B7CA92F04B9FAE8D941C3E9B7E0CD754'


In [ ]:
guids = [
    "032A5E114A59D28267BDD8B6D9E58B22",
    "21BCCD6C4110696EABE622B71D1C0566",
    "25D40AD444B6D51B333021ADFB219501",
    "293664374424ADAC7D3615AE3B384907",
    "2B028FBC466C3DC839E1E092FDF4CC56",
    "2F2302AE4D1C2DDDFA9CDCB46802B185",
    "31FF880045D230BE04C209A42C108B24",
    "3673EC9146EE4DF3517725BBA97A4B36",
    "3A0BE6C741DE7CDD4A4C01925B5A04E9",
    "3E312E64489040609C0A63800A4E3BAE",
    "5D8A265D4A08641AAB76DAB74440A41C",
    "7065AA4C48A2044AF0BEED95042A4870",
    "745D4D304697491769F4F4B41E906BE0",
    "74F9B2164869627AF637FDA5D4A121B3",
    "7576CD5F48607C21C914ACBE053B259B",
    "7F16A4B811E58ED317D50080EFF554EA",
    "92ADD0F84D07AC532AD03BA0F92A836B",
    "96648F2B492150A6AA27DDB3744E32B4",
    "986E6D1A427030A710A0E68F74290482",
    "9981CB214C79D75EC342A781539A4E5A",
    "9A96A6474DB13EA45D4B96B4485463C8",
    "9BB06A4C43974CEF178D2D9843018297",
    "A8B8EF0949433604826137AC569859E3",
    "AFA89D9C4313375DC9945AA4634C5FED",
    "B70AF11248C00F7D4339079310F8B3F1",
    "B7AE522248AD2604FA45489C1B750FBD",
    "B7CA92F04B9FAE8D941C3E9B7E0CD754",
    "BA38E83548881BF83570CCABE2CB9B4B",
    "BB1630284A9D1EA06DCF69A1A57BD24B",
    "BF03DD1443947C6192D0229B4A3D7EF8",
    "D1D2CE9E43C95905EF6640AA0576FE1C",
    "D41FC55D4282E9A5C943D9A8670336CF",
    "D63BE643469DE5784DD83C8F95B8BFE6",
    "D64C532E4E7FBA74D29A7CA3576F39CF",
    "E098E8DD49B73D9FBE4F17ACC6774F3E",
    "E6A7C8EE4BB767E8380E3A896092C0E6",
    "EF66D49D40D0E799E1D1909A65085E97",
    "F025A94B4C03B6DCEE0F5D9DA825DA67",
]

guids.append("CF2049E94A0000A16532D39C6D783F1D")
guids.append("9981CB214C79D75EC342A781539A4E5A")
guids.append("9A3D2E3C49EA56FEAC9231919DBC51BF")
guids.append("AC7CC55242A3512A1709BD8EB9A7A73A")
guids.append("410F74B843A27E1BB7BF9C8891EA8D0C")
guids.append("14B57F1F46127BC6E6F746888BC3C7CB")
guids.append("1F21042E4FC08886A9D7E4920CA4EC59")







In [36]:
from mstr_robotics._connectors import MstrApi

i_MstrApi= MstrApi()
mig_obj_li= i_MstrApi.get_proj_obj_by_id_l(conn=conn,obj_id_l=guids)
mig_obj_li
mig_list=[]
for obj in mig_obj_li:
    mig_d={
        "object_guid": obj.get("id"),
        "name": obj.get("name"),
        "type": obj.get("type")
    }
    mig_list.append(mig_d)
mig_list_df=pd.DataFrame(mig_list)
mig_list_df.to_excel("C:\\Users\\danie\\Downloads\\mig_list.xlsx")

In [29]:
conn.select_project("7576CD5F48607C21C914ACBE053B259B")
pa_mig_obj_li= i_MstrApi.get_proj_obj_by_id_l(conn=conn,obj_id_l=guids)
pa_mig_list=[]
for obj in pa_mig_obj_li:
    mig_d={
        "object_guid": obj.get("id"),
        "name": obj.get("name"),
        "type": obj.get("type")
    }
    pa_mig_list.append(mig_d)
pa_mig_list_df=pd.DataFrame(pa_mig_list)

pa_mig_list_df.to_excel("C:\\Users\\danie\\Downloads\\pa_mig_list.xlsx")

## Endpoints

In [13]:
def get_child_objects(conn,object_id):
    # Both helpers are delegated to the mstr_robotics package:
    # - i_mstr_api.get_proj_obj_by_id_l: resolves the object's type/subtype/name
    #   from just the id (same searches/objects endpoint the old get_obj_details used).
    # - i_md_searches.search_for_used_in_obj_direct: wraps store_search_instance +
    #   paged get_search_results to find the dependents, handling the 0-results case.
    obj_det_l=i_mstr_api.get_proj_obj_by_id_l(conn,[object_id],org_def_fg=True)
    if not obj_det_l:
        return []
    object_det_d=obj_det_l[0]
    obj_l=[{"id":object_det_d["id"],"type":object_det_d["type"]}]
    dpn_rows=i_md_searches.search_for_used_in_obj_direct(
        conn,obj_l,dpn_fg=True,info_level="base",count_only_fg=False)
    dpn_child_d_l=[]
    for row in dpn_rows:
        # objects with no dependents yield a dummy row without dpn_ keys; skip it
        if not row.get("dpn_id"):
            continue
        dpn_child_d={}
        dpn_child_d["id"]=object_det_d["id"]
        dpn_child_d["type"]=object_det_d["type"]
        dpn_child_d["subtype"]=object_det_d["subtype"]
        dpn_child_d["name"]=object_det_d["name"]
        dpn_child_d["child_dpn_id"]=row["dpn_id"]
        dpn_child_d["child_dpn_type"]=row["dpn_type"]
        dpn_child_d["child_dpn_subtype"]=row["dpn_subtype"]
        dpn_child_d["child_dpn_name"]=row["dpn_name"]
        dpn_child_d_l.append(dpn_child_d.copy())
    return dpn_child_d_l

In [ ]:
#object ids are maintained in ..\config\jupyter_objects_d.yml
with open("..\\config\\jupyter_objects_d.yml", "r") as openfile:
    jupyter_objects_d = yaml.safe_load(openfile)
nb_d = jupyter_objects_d["semantic_endpoints"]

#metric
metric_d_l=[]
metric_id=nb_d["misc"]["metric_id"]
metric_l=[metric_id]
for metric_id in metric_l:
    metric_def=metrics.get_metric(
        connection=conn,
        id=metric_id,
        changeset_id=None,
        show_expression_as="tokens",
        show_filter_tokens=False
    )
    metric_def=metric_def.json()
    metric_d={}
    metric_d["project_id"]=conn.project_id
    metric_d["id"]=metric_id
    metric_d["name"]=metric_def["name"]
    metric_d["text"]=metric_def["expression"]["text"]
    metric_d["expression"]=metric_def["expression"]
    metric_d["transformations_l"]=None
    metric_d["commplexity"]=1
    metric_d_l.append(metric_d.copy())
    all_comp_obj_d_l.append(get_child_objects(conn,object_id=metric_id))

metric_d_l


In [ ]:
#filter
filter_d_l=[]
filter_l=nb_d["misc"]["filter_l"]
for filter_id in filter_l:
    filter_def=filters.get_filter(connection=conn, id=filter_id, project_id=project_id).json()
    filter_d={}
    filter_d["project_id"]=conn.project_id
    filter_d["id"]=filter_id
    filter_d["name"]=filter_def["name"]
    filter_d["text"]=filter_def["qualification"]["text"]
    filter_d["filter_type"]=filter_def["qualification"]["tree"]["type"]
    filter_d["commplexity"]=1
    filter_d["qualification"]=filter_def["qualification"]
    filter_d_l.append(filter_d.copy())
    all_comp_obj_d_l.append(get_child_objects(conn,object_id=filter_id))
filter_d_l

In [ ]:
#attributes
attribute_d_l=[]   
att_parent_child_d_l=[] 
attribute_l=nb_d["misc"]["attribute_l"]
attribute_def=attributes.get_attribute(connection=conn,id=attribute_l[0],show_expression_as="tokens").json()

for att in attribute_l:
    attribute_d={}
    attribute_d["project_id"]=conn.project_id
    attribute_def=attributes.get_attribute(connection=conn,id=att,show_expression_as="tokens").json()
    attribute_d["id"]=attribute_def["id"]
    attribute_d["name"]=attribute_def["name"]
    #attribute_d["text"]=attribute_def["expression"]["text"]
    attribute_d["def"]=attribute_def
    attribute_d_l.append(attribute_d.copy())


for r in attribute_def["relationships"]:
  
    att_parent_child_d={}

    if attribute_def["id"]!=r["parent"]["objectId"]:

        att_parent_child_d["rel_attribute_id"]=r["parent"]["objectId"]
        att_parent_child_d["rel_attribute_name"]=r["parent"]["name"]
        att_parent_child_d["rel_table"]=r["relationshipTable"]["name"]
        att_parent_child_d["rel_type"]=r["relationshipType"]
        att_parent_child_d["type"]="parent"
    else:

        att_parent_child_d["rel_attribute_id"]=r["child"]["objectId"]
        att_parent_child_d["rel_attribute_name"]=r["child"]["name"]
        att_parent_child_d["rel_table"]=r["relationshipTable"]["name"]
        att_parent_child_d["rel_type"]=r["relationshipType"]
        att_parent_child_d["type"]="child"
    att_parent_child_d_l.append(att_parent_child_d.copy())
att_parent_child_d_l

In [17]:
#facts

In [ ]:
#transformations
transformation_d_l=[]   
transformation_l=nb_d["misc"]["transformation_l"]
for t in transformation_l:
    transformation_d={}
    trans_def=transformations.get_transformation(connection=conn,id=t).json()
    transformation_d["id"]=trans_def["id"]
    transformation_d["name"]=trans_def["name"]
    transformation_d["mapping_type"]=trans_def["mappingType"]
    transformation_d["def"]=trans_def
    transformation_d_l.append(transformation_d.copy())


transformation_d_l

In [ ]:
#Olap reports
report_d_l=[]
report_l=nb_d["reports"]["report_l"]
for report_id in report_l:
    all_comp_obj_d_l.append(get_child_objects(conn,object_id=report_id))

In [ ]:
#OlapCubes
olap_cube_l=nb_d["cubes"]["olap_cube_l"]
report_l=nb_d["reports"]["report_l"]
for cube_id in olap_cube_l:
    all_comp_obj_d_l.append(get_child_objects(conn,object_id=cube_id))